In [ ]:
from itertools import zip_longest as zip 
from itertools import chain
from itertools import product

"""
A_q(lambda) of Symplectic group group. 

The weight lattice is Z^n for Sp(2n,R).
We will use half integral weights. 
In the following tuple of integer represents the 2 * weight. 
"""

def tworhoA(n):
    return list(n - 2*i - 1for i in range(n))


def tworhoC(n):
    return list(2*n - 2*i for i in range(n))

def smul_list(s,a):
    return [s * x for x in a]

def sum_list(a, b):
    """
    The sum of tuples a and b is given by
    c[i] = a[i] + b[i]
    """
    a,b = list(a),list(b)
    assert len(a) == len(b), "Tuples must have the same length"
    return [x + y for x, y in zip(a, b)]


def sub_list(a, b):
    """
    The sum of tuples a and b is given by
    c[i] = a[i] - b[i]
    """
    a,b = list(a),list(b)
    assert len(a) == len(b), "Tuples must have the same length"
    return [x - y for x, y in zip(a, b)]

"""
The theta-stable q is defined by Q := ([(p_i,q_i)],n_0) with Levi U(p_1,q_1) x ... x U(p_k,q_k) x Sp(2n_0) 
The A_q(lambda) is given by (Q,[l_1, ... ,l_k]), where l_i denote the character of $U(p_i,q_i)\to U(1)$ and there is no non-trivial character on the Sp(2n_0).  
"""

"""
The list [(p_i,q_i)] give a compact L if and only if p_i q_i = 0 for all i. 
This is the case when A_q(lambda) can be a discrete series. 
"""
def is_Lq_discrete(Q): 
    Q , n = Q
    return all(p*q==0 for p,q in Q) and n == 0 


def sizeQC(Q):
    Q, n = Q
    return n+sum(p+q for p,q in Q)

def signQC(Q):
    """
    For Q = ([(p_i,q_i)], n_0) it gives sequence of sign + ... + - ... - +...+ -...- with +^p_1 -^q_1 ... +^p_k -^q_k +^n_0
    """
    Q, n = Q
    Q += [(n,0)]
    res = []
    for p,q in Q:
        res += [1]*p + [-1]*q
    return res

def sign2Q(s):
    """
    From sign sequence to a Q = [(p_1,0), (0,q_2),(p_3,0),...,] or [(0,q_2),(p_3,0),...,]
    with p_i, q_i >0.
    """
    if len(s) == 0 : return []
    ce = s[0]
    c = 0
    s += -1*s[-1]
    res = []
    for e in s:
        if e == ce:
            c += 1
        if e!= ce:
            res.append((c,0) if ce == 1 else (0,c)) 
            ce, c = e, 1 
    return res


def sign2QC(s):
    return (sign2Q(s),0)


def tworholC(Q):
    Q, n = Q
    return list(chain(*[tworhoA(p+q) for p,q in Q])) + tworhoC(n) 


def tworhouC(Q): 
    """
    The function compute rho(u)
    We use rho(u) = rho - rho_L where rho_L is the rho of the Levi
    """
    return sub_list(tworhoC(sizeQC(Q)),tworholC(Q))

def PM2Sign(s):
    """
    From +- sequence to sign 
    """
    return [1 if e == '+' else -1 for e in s] 

def Sign2PM(s):
    return ''.join('+' if e == 1 else '-' for e in s)

def w0gamma(gamma):
    """
    The coxeter element w0 action on gamma sends [g_1,...,g_n] to [-g_n,..., g_1]
    """
    return [-g for g in gamma[::-1]]

def tworhokC(Q):
    """
    For Q = ([(p_i,q_i)], n_0) it gives sequence of sign + ... + - ... - +...+ -...- with +^p_1 -^q_1 ... +^p_k -^q_k +^n_0
    //Let Q' = ([(p'_i,q'_j)]) such that p'_i * q'_i = 0 be the data give the same sequence.
    The positive roots of K are (e_i-e_j) if i,j has the same sign or (e_i + e_j) if i,j has different sign with i<j 
    """
    # generate the sign list 
    s = signQC(Q)
    # the group is Sp(2n)
    n = len(s)
    p,q = 0,0 
    res = []
    for i in range(n):
        res += [n-2*i-1+2*q] if s[i]==1 else  [n-2*i-1+2*p]
        p, q  = (p+1,q) if s[i] == 1 else (p,q+1)
    return res

def tworholkC(Q):
    """
    2 * rho_(l \cap k)
    """
    Q, n = Q
    Q += [(n,0)]
    res = []
    for pi,qi in Q:
        res = res + tworhoA(pi)+tworhoA(qi)
    return res

def tworhoukC(Q):
    """
    rho_(u \cap k) = rho_k - rho_{l\cap k}
    """
    return sub_list(tworhokC(Q),tworholkC(Q))

def tworhoupC(Q):
    """
    rho_(u \cap p) = rho_u - rho_{u\cap k}
    """
    return sub_list(tworhouC(Q),tworhoukC(Q))


"""
A complete L-paramter of Unitary group or rank n is a pair of a list of numbers (a_1, ...,  a_n) and 
a list of sign (e_1, ..., e_n)
"""


"""
In the following assume A_q(lambda) is a discrete series. We 
The Harish-Chandra parameter of A_q(lambda) is nothing but lambda+rho with respect to a postive system compatible with q. 
The missing data is the choice of positive system. 
"""

def lamhC(Q,lam):
    """
    Give a list of signature, lam a list of integers
    lamh = l_1 ... l_1 l_2 ... l_2 ... l_k ... l_k 0...0 where l_i occure p_i+q_i times and 0 occure n_0 times. 
    """
    Q, n = Q
    assert(len(Q) == len(lam))
    res = []
    for (pi,qi),l in zip(Q,lam):
        res += [l]*(pi+qi)
    res += [0]*n
    return res

def changecod(Q,gamma):
    NQ = sign2Q(signQC(Q))
    resp,resq = [], []
    for p,q in NQ:
        resp += gamma[:p]
        gamma = gamma[p:]
        resq += gamma[:q]
        gamma = gamma[q:]
    
    return resp+w0gamma(resq)
    

def HCParameterC(Q,lam=None):
    """
    Harisch-Chandra paramter as weight of Sp(2n) 
    Here we need to do a change of variable in U(n):
    For (g_1,g_2, ..., g_n) with sign + ... + -...- +...+ -...- , it produce 
    (g_1, ... g_p_1, -g_(p_1+q_1)... -g_(p_1), g_(p_1+q_1+1) ... g_(p_1+q_1+p_2)...)
    """
    if not lam:
        lam = [0]*len(Q[0])
    n = sizeQC(Q)
    gamma = sum_list(lamhC(Q,lam),tworhoC(n))
    return tuple(changecod(Q,gamma))

def MinimalKtypeC(Q,lam=None):
    """
    It is given by lam + 2rho_up
    When A_q(lam) is a discrete series it is also called the Blattner paramter.
    """    
    if not lam:
        lam = [0]*len(Q[0])
    Lam = sum_list(lamhC(Q,lam),smul_list(2,tworhoupC(Q)))
    return tuple(changecod(Q,Lam)) 


In [ ]:
"""
Give n generate all possible signs 
"""
def all_signs(n):
    return product([1,-1],repeat=n)

def sign2Q(s):
    """
    give a tuple of sign, return the Q for discrete series. 
    Break a tuple of +1 and -1 into (p,0) and (0,q) tuples.
    Args:
    input_tuple (tuple): A tuple containing +1 and -1 values.
    
    Returns:
    list: A list of tuples (p,0) and (0,q) representing consecutive +1's and -1's.
    """
    if len(s) == 0: return []
    res = []
    count = 0
    current = s[0] 

    for value in chain(s, [-1*s[-1]]):
        if value != current:
            res.append((count, 0) if current == 1 else (0, count))
            current = value
            count = 1
        else:
            count += 1

    return res



In [ ]:
"""
Signed Young diagram is give by a list [(l_i,p_i,q_i)] where l_i is the length and p_i, q_i are the signature. 
"""
def regSYD(SY):
    res = dict()
    for l,p,q in SY:
        p0,q0 = res.get(l,(0,0))
        res[l] = p0+p,q0+q
    res = [(l,p,q) for l,(p,q) in res.items() if p!=0 or q!=0]
    res = sorted(res,key=lambda x: x[0],reverse=True)
    return res

def pmstr(n,e):
    if e == 1:
        return '+-'*(n//2)+'+'*(n%2)
    else:
        return '-+'*(n//2)+'-'*(n%2)


def strSYD(SY):
    #SY = sorted(SY,key = lambda x: x[0], reverse=True)
    SY = regSYD(SY)
    SY = sorted(SY, reverse=True)
    res = ''
    for l,p,q in SY:
        res += '\n'.join([pmstr(l,1)]*p+[pmstr(l,-1)]*q)+'\n'
    return res

def addright(SY,p,q):
    """
    Add signs on the right of the diagram
    """
    res = []
    SY = regSYD(SY)
    for l,pi,qi in SY: 
        # in the following ri, si represent number 
        # of + and - sign adding each time
        if l%2 == 0 :
            # even row add (+,-)
            ri ,si = min(pi,p),min(qi,q)
            res.append((l+1,ri,si))
            res.append((l,pi-ri,qi-si))
        else:
            # odd row add (-,+)
            si ,ri = min(pi,q),min(qi,p)
            res.append((l+1,si,ri))
            res.append((l,pi-si,qi-ri))
        p -= ri
        q -= si
    res.append((1,p,q))
    return regSYD(res)

def addleft(SY,p,q):
    """
    Add signs on the left of the diagram
    """
    res = []
    SY = regSYD(SY)
    for l,pi,qi in SY: 
        # in the following ri, si represent number 
        # of + and - sign adding each time

        # add (-,+) 
        # note that the sign of the row should be change
        ri ,si = min(pi,q),min(qi,p)
        res.append((l+1,si,ri))
        res.append((l,pi-ri,qi-si))
        p -= si
        q -= ri
    res.append((1,p,q))
    return regSYD(res)

def AVofAqC(Q):
    """
    The AV only depends on Q
    """
    Q,n = Q
    SY = [(1,n,0)]
    for p,q in reversed(Q):
        SY = addright(SY,p,q) 
        SY = addleft(SY,q,p)
    return SY


In [ ]:
""" Test """
Q = ([(2,0),(0,3),(4,0),(0,2),(2,0)],0)
QQ = ([(1,0),(1,0),(0,2),(0,1),(2,0),(2,0),(0,2),(2,0)],0)


print("rho_u", tworhouC(Q))
print("rho_k", tworhokC(Q))

print("rho_up",tworhoupC(Q))
print("rho_up",tworhoupC(QQ))



In [ ]:
print("rho_up",tworhoupC(([(2,0),(2,0),(0,2),(3,0)],0)))
print("rho_up",tworhoupC(([(4,0),(0,2),(3,0)],0)))

In [ ]:
n=5
SS = list(all_signs(n))
print(list(all_signs(n)))
BP = set() 
for s in SS:
    print('-'*20)
    Q = sign2QC(s)
    print(f"Sign parameter {Sign2PM(s)}")
    #print(AVofAq(Q))
    print(strSYD(AVofAqC(Q)))
    print(f"2rho_k: {tworhokC(Q)}")
    assert(changecod(Q,tworhokC(Q))==tworhoA(sizeQC(Q)))
    hp = HCParameterC(Q)
    print(f"HC parameter: {hp}")
    #print(sizeQ(Q),Q)
    b = MinimalKtypeC(Q)
    print(f"Blattner :{b}")
    BP.add(b)

assert(len(BP)==2**n)

In [ ]:
SY = [(3,2,1),(2,1,2),(5,3,0),(5,0,3),(5,1,2),(4,0,3)]
print(strSYD(SY))
